# Phase 1 — SimCLR reference encoder, seed 1 (Colab)

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = "/content/drive/MyDrive/probe-capacity-results/encoders"
import os; os.makedirs(DRIVE_OUT, exist_ok=True)
print("Drive mounted. Output dir:", DRIVE_OUT)

## 2. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/content/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime > Change runtime type > T4 GPU")

## 4. Download Shapes3D + build the image cache

In [ ]:
!cd /content/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 4b. Restore checkpoint from Drive (optional — only if resuming a previous run)

In [ ]:
import shutil
from pathlib import Path

RUN_ID = "standard_strong_seed1"
drive_ckpt = Path(DRIVE_OUT) / RUN_ID / "last_ckpt.pt"
local_ckpt = Path("results/encoders") / RUN_ID / "last_ckpt.pt"

if drive_ckpt.exists():
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(drive_ckpt, local_ckpt)
    print(f"restored {drive_ckpt} -> {local_ckpt}")
else:
    print(f"no checkpoint found at {drive_ckpt} — starting from epoch 0")

## 5. Train the reference encoder — seed 1

In [ ]:
!cd /content/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=1 run.num_workers=2

## 6. Sync results to Google Drive

In [ ]:
import json, shutil
from pathlib import Path

REPO_DIR = "/content/probe-capacity-invariance"
RUN_ID   = "standard_strong_seed1"
src_dir  = Path(REPO_DIR) / "results" / "encoders" / RUN_ID
dst_dir  = Path(DRIVE_OUT) / RUN_ID
dst_dir.mkdir(parents=True, exist_ok=True)

if not src_dir.exists():
    print(f"nothing to sync yet — {src_dir} not created "
          "(training hasn't finished epoch 0). Re-run after the first epoch.")
else:
    for f in sorted(src_dir.iterdir()):
        if f.is_file():
            shutil.copy2(f, dst_dir / f.name)
            print(f"synced {f.name} ({f.stat().st_size // 1024} KB)")

    log = src_dir / "train_log.jsonl"
    if log.exists():
        lines = [ln for ln in log.read_text().splitlines() if ln.strip()]
        if lines:
            latest = json.loads(lines[-1])
            (dst_dir / "metrics_latest.json").write_text(json.dumps(latest, indent=2))
            print(f"wrote metrics_latest.json (epoch {latest.get('epoch')})")
    print(f"done -> {dst_dir}")

## 7. Verify final output

In [ ]:
import json
from pathlib import Path

RUN_ID = "standard_strong_seed1"
drive_dir = Path(DRIVE_OUT) / RUN_ID

for fname in ["backbone.pt", "metrics.json", "train_log.jsonl", "last_ckpt.pt"]:
    p = drive_dir / fname
    status = f"{p.stat().st_size // 1024} KB" if p.exists() else "MISSING"
    print(f"{fname}: {status}")

metrics_path = drive_dir / "metrics.json"
if metrics_path.exists():
    print()
    print(json.dumps(json.load(open(metrics_path)), indent=2))

## 8. Encoder-quality gate (Phase-1 exit)

In [ ]:


import shutil
from pathlib import Path
from google.colab import files

REPO = Path("/content/probe-capacity-invariance")

def ensure_backbone(run_id):
    dst = REPO / "results" / "encoders" / run_id / "backbone.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        print(f"✓ {run_id}: already present ({dst.stat().st_size // 1024 // 1024} MB)")
        return
    drive_src = Path(DRIVE_OUT) / run_id / "backbone.pt"
    if drive_src.exists():
        shutil.copy2(drive_src, dst)
        print(f"✓ {run_id}: restored from Drive")
        return
    print(f"↑ {run_id}: not on disk or Drive — upload ITS backbone.pt now:")
    up = files.upload()
    name = next(iter(up))
    Path(name).replace(dst)
    print(f"✓ {run_id}: placed uploaded {name}")

for rid in ["standard_strong_seed0", "standard_strong_seed1"]:
    ensure_backbone(rid)

In [ ]:


!cd /content/probe-capacity-invariance && python -m src.encoders.supervised \
    --config configs/run/supervised.yaml \
    --set run.seed=0 run.num_workers=2

In [ ]:



!cd /content/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/reference_cuda.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --supervised results/encoders/supervised_seed0/backbone.pt \
    --random-seed 0 \
    --out results/quality_gate/reference.json

In [ ]:

import shutil
from pathlib import Path

REPO = Path("/content/probe-capacity-invariance")
RESULTS_ROOT = Path(DRIVE_OUT).parent

gate_src = REPO / "results" / "quality_gate" / "reference.json"
gate_dst = RESULTS_ROOT / "quality_gate" / "reference.json"
gate_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(gate_src, gate_dst)
print(f"synced {gate_dst}")


sup_src = REPO / "results" / "encoders" / "supervised_seed0" / "backbone.pt"
if sup_src.exists():
    sup_dst = Path(DRIVE_OUT) / "supervised_seed0" / "backbone.pt"
    sup_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sup_src, sup_dst)
    print(f"synced {sup_dst}")

print("\n--- reference.json ---")
print(gate_src.read_text())